# 02b · Feature engineering (II): qué entra al modelo y cómo lo preparamos

En `02a_features_cliente` montamos una tabla con una fila por cliente y un montón de variables candidatas. Ahora toca decidir **cuáles entran** al K-Means y **prepararlas** para que el algoritmo las pueda usar bien.

K-Means agrupa por distancias, así que es muy sensible a tres cosas: que haya nulos (directamente no funciona), que las variables estén en escalas distintas (la que tenga números más grandes manda) y que haya valores extremos (un cliente con 561 coches arrastra un centroide entero). Todo lo que hacemos aquí va de eso.

Al final lo metemos todo en un `Pipeline` de sklearn, que es lo que pide el reto y lo que vamos a reutilizar en el reto 3 y en el Jackknife.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config, clean
from src import pipeline as pl

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Partimos de la tabla de clientes

La genera el notebook `02a`. Si alguien no lo ha ejecutado, la construimos aquí directamente (tarda unos segundos).

In [ ]:
if config.CLIENTS_PKL.exists():
    clientes = clean.load_clients()
else:
    clientes = clean.build_client_table(clean.load_raw())
clientes.shape

## 2. Qué entra y qué no

### 2.1 Lo que usamos solo para describir

Hay columnas que tenemos en la tabla pero que **no queremos que decidan los segmentos**:

- **Género:** está inferido, no declarado, y le falta al 17 %. Segmentar por eso no nos parece ni fiable ni ético.
- **Edad en texto, país en texto, clase exacta, grupo de producto:** ya están representados por otras columnas (`age_ord`, `is_ch`, `main_class_group`...).
- **`is_heavy_account`:** es un corte a mano de `n_vehicles`, que ya entra.
- **Consentimientos:** este merece una explicación.

In [ ]:
clientes[["consent_phone", "consent_epost", "consent_post"]].mean().round(3).to_frame("% clientes que aceptan")

El email lo acepta el 98 % y todos tienen al menos un canal abierto. Pero el motivo principal para dejarlos fuera no es ese: los consentimientos dicen **por dónde podemos contactar** a alguien, no **cómo es**. Queremos segmentos definidos por el cliente (su coche, su relación con la marca, su zona) y luego, para cada segmento, mirar por qué canal se le habla. Así los usaremos en el reto 4.

### 2.2 Telemetría: fuera del modelo base

Los datos de uso (distancia por viaje, velocidad, consumo, horas) son muy interesantes, pero:

In [ ]:
tele = ["trip_dist_avg", "speed_avg", "fuel_cons_avg", "op_hours_mean"]
clientes[tele].isna().mean().mul(100).round(1).to_frame("% clientes sin dato")

Falta en casi la mitad de los clientes. Y no por error: son clientes sin coche conectado. Si rellenáramos esos huecos con la mediana, la mitad de la base caería exactamente en el mismo punto en esas cuatro dimensiones, y K-Means lo vería como un grupo "real". Nos estaríamos inventando el comportamiento de conducción de unas 87.000 personas.

Así que en el modelo base la telemetría entra solo como flag (`has_telemetry`: el cliente está conectado o no). Dejamos preparada una **variante con telemetría** para compararlas en el reto 3 y ver si de verdad aporta.

### 2.3 Variables que dicen lo mismo

Si dos variables están muy correlacionadas, meter las dos es como contar dos veces lo mismo: le damos el doble de peso a esa idea sin querer. Miramos las candidatas (usamos Spearman porque muchas no son lineales):

In [ ]:
candidatas = ["age_ord", "tenure_months", "ppi", "population", "n_connected_now", "n_vehicles",
              "main_vehicle_age", "fleet_age_mean", "workshop_visits_mean", "share_plugin_ev",
              "is_ch", "is_b2b", "has_current_vehicle", "tenure_is_default", "has_telemetry"]
corr = clientes[candidatas].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 7})
plt.title("Correlación (Spearman) entre variables candidatas")
plt.show()

In [ ]:
pares = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().rename("rho").rename_axis(["variable 1", "variable 2"]).reset_index())
pares.reindex(pares.rho.abs().sort_values(ascending=False).index).head(6).round(2)

Hay tres parejas que nos obligan a decidir:

1. **`main_vehicle_age` y `fleet_age_mean` (0,89).** Para el 73 % de clientes que tienen un solo coche son literalmente el mismo número. Nos quedamos con la antigüedad del **vehículo principal**, que es la que importa para saber si le toca renovar.
2. **`n_connected_now` y `has_current_vehicle` (0,78).** El flag es simplemente "tiene más de 0 vehículos conectados". Nos quedamos con el número, que contiene al flag. El flag lo guardamos para describir a los clientes inactivos.
3. **`ppi` e `is_ch` (0,81).** Esta es la más interesante, la vemos aparte.

### 2.4 Poder adquisitivo relativo a su país → `ppi_rel`

In [ ]:
clientes.groupby("country_code").ppi.median().round(0).to_frame("ppi mediano (€)")

En Suiza el poder adquisitivo mediano por habitante es de unos 51.000 € y en Austria de unos 30.000 €. Así que `ppi` en realidad nos está diciendo sobre todo "este cliente es suizo", cosa que ya sabemos por `is_ch`.

Lo que nos interesa para marketing es otra cosa: **si el cliente vive en una zona rica o pobre dentro de su mercado**. Un austriaco en una zona de 45.000 € es un cliente acomodado aunque esté por debajo de la media suiza. Así que creamos `ppi_rel`: el poder adquisitivo de su zona dividido entre la mediana de su país (1 = lo típico de su país, 1,5 = un 50 % por encima).

In [ ]:
mediana_pais = clientes.groupby("country_code").ppi.transform("median")
ppi_rel = clientes.ppi / mediana_pais
print("Correlación con is_ch:")
print(f"  ppi:     {clientes.ppi.corr(clientes.is_ch, method='spearman'):.2f}")
print(f"  ppi_rel: {ppi_rel.corr(clientes.is_ch, method='spearman'):.2f}")

Pasa de 0,81 a 0,03. Ahora el país va en `is_ch` y la riqueza relativa en `ppi_rel`, cada idea en su variable. En el pipeline lo calcula un transformer nuestro (`CountryMedianImputer`) que aprende las medianas de cada país al entrenar.

### 2.5 Lista de features del modelo

Con todo esto, las features que entran al modelo base quedan así (están definidas en `src/pipeline.py`):

In [ ]:
bloques = {"con cola larga (log)": pl.LOG_COLS, "numéricas": pl.NUM_COLS,
           "binarias": pl.BIN_COLS, "categóricas": pl.CAT_COLS}
for nombre, cols in bloques.items():
    print(f"{nombre:22s} {cols}")

## 3. Nulos

Miramos qué nulos quedan en las features elegidas (`ppi_rel` se calcula a partir de `ppi`, así que miramos `ppi`):

In [ ]:
usadas = [("ppi" if c == "ppi_rel" else c) for c in pl.LOG_COLS + pl.NUM_COLS + pl.BIN_COLS + pl.CAT_COLS]
nulos = clientes[usadas].isna().mean().mul(100).round(2)
nulos[nulos > 0].sort_values(ascending=False).to_frame("% nulos")

Cómo los tratamos y por qué:

- **`share_plugin_ev` (15 %):** clientes de los que no conocemos el motor de ningún coche. Los rellenamos con la mediana, que es 0: sin pruebas de que tenga un enchufable, lo tratamos como la mayoría (de los clientes con motor conocido, tres de cada cuatro no tienen ninguno).
- **`ppi` y `population` (9 %):** clientes cuyo código postal no tiene datos socioeconómicos. Los rellenamos con la **mediana de su país**, no con la global. Si no, a un austriaco le pondríamos un valor medio entre Suiza y Austria que no corresponde a ninguno. Con la población pasa algo parecido: las zonas típicas de un país y otro no tienen el mismo tamaño.
- **`main_vehicle_age` (1 %) y `tenure_months` (2 clientes):** mediana. Son tan pocos que cualquier criterio razonable da lo mismo.

Un detalle importante: los nulos los rellenamos **dentro del pipeline**, no antes. Así las medianas se aprenden al entrenar, y cuando en el Jackknife re-entrenemos con un 85 % de los clientes, se recalculan con esa submuestra, como tiene que ser.

## 4. Variables con cola larga → logaritmo

Muchas variables de conteo tienen casi todos los clientes en valores bajos y unos pocos con valores enormes. Medimos la asimetría (skew) antes y después de aplicar `log(1 + x)`:

In [ ]:
num_cand = ["n_vehicles", "n_connected_now", "population", "workshop_visits_mean",
            "age_ord", "tenure_months", "main_vehicle_age", "share_plugin_ev"]
asim = pd.DataFrame({"skew original": clientes[num_cand].skew(),
                     "skew con log1p": np.log1p(clientes[num_cand]).skew()})
asim.loc["ppi_rel"] = [ppi_rel.skew(), np.log1p(ppi_rel).skew()]
asim.round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for fila, col in enumerate(["population", "workshop_visits_mean"]):
    axes[fila, 0].hist(clientes[col].dropna(), bins=60, color="steelblue")
    axes[fila, 0].set_title(f"{col} (original)")
    axes[fila, 1].hist(np.log1p(clientes[col].dropna()), bins=60, color="seagreen")
    axes[fila, 1].set_title(f"{col} (log1p)")
plt.tight_layout()
plt.show()

El criterio que seguimos: **logaritmo solo donde arregla la asimetría**.

- `n_vehicles`, `n_connected_now`, `population`, `workshop_visits_mean` y `ppi_rel` mejoran muchísimo (la población pasa de 3,6 a 0,1, por ejemplo). Llevan log.
- `n_vehicles` sigue asimétrica incluso con log (4,3) por las flotas. Eso lo arreglamos en el siguiente paso.
- `age_ord`, `tenure_months` y `main_vehicle_age` ya eran bastante simétricas, y el log las **empeora** (la antigüedad pasa de 0,4 a -1,7). Se quedan como están.
- `share_plugin_ev` es una proporción entre 0 y 1; el log apenas la cambia. Se queda igual.

## 5. Valores extremos → recorte p1-p99

In [ ]:
clientes.n_vehicles.quantile([0.5, 0.9, 0.99, 0.999, 1]).to_frame("nº vehículos")

El 99 % de los clientes tiene 10 vehículos o menos, pero el máximo es 561. En K-Means un punto así tira del centroide de su cluster hacia él, y puede acabar teniendo un cluster casi para él solo.

Por eso recortamos cada variable numérica a sus **percentiles 1 y 99**: lo que esté por encima del p99 se queda en el p99. En `n_vehicles` eso significa que cualquier cliente con 10 o más coches se trata como 10. Seguimos sabiendo que son clientes "grandes", pero ya no deforman el modelo. Los límites los aprende el transformer `QuantileClipper` al entrenar, igual que las medianas.

En el EDA también vimos valores raros en la telemetría (velocidades medias de más de 150 km/h, consumos de 50 l/100 km). El mismo recorte se los come en la variante con telemetría.

## 6. Categóricas y escalado

- **Categóricas sin orden** (`main_class_group`, `main_engine`, `finance_leasing_status`): one-hot. Un compacto no es "menos" que una furgoneta; si las codificáramos como 1, 2, 3... K-Means se inventaría distancias entre ellas. La edad no entra aquí porque sí tiene orden y ya la pasamos a 0-5 en el `02a`.
- **Numéricas**: `StandardScaler` (media 0, desviación 1), para que ninguna mande solo por tener números más grandes. Sin esto, la población (miles) aplastaría a la edad (0-5).
- **Binarias y one-hot**: las dejamos en 0/1. Así cada una pesa algo menos que una numérica escalada, y es a propósito: entre one-hot y flags tenemos 20 columnas binarias frente a 9 numéricas, y no queremos que el motor o la clase del coche acaparen todo el modelo.

## 7. Montamos el pipeline

Todo lo anterior va encadenado en un `Pipeline` de sklearn (`pl.build_preprocessor()`):

In [ ]:
prep = pl.build_preprocessor()
prep

In [ ]:
X = prep.fit_transform(clientes)
nombres = prep.get_feature_names_out()

print("Forma de X:", X.shape)
print("Nulos:", np.isnan(X).sum(), "| Infinitos:", np.isinf(X).sum())

In [ ]:
pd.DataFrame(X, columns=nombres).agg(["mean", "std", "min", "max"]).T.round(2)

Todo en orden: 173.456 clientes y 29 columnas, sin nulos ni infinitos. Las numéricas salen con media 0 y desviación 1, las binarias y one-hot entre 0 y 1. Las 29 columnas son las 9 numéricas + 4 flags + 7 clases + 5 motores + 4 situaciones de financiación.

Las medianas por país que aprendió el primer paso:

In [ ]:
prep.named_steps["pais"].medians_

## 8. Un pipeline que va del CSV a X

El reto pide un pipeline que **encapsule todo el proceso de transformación**. El de arriba empieza en la tabla de clientes, así que le ponemos delante la construcción de esa tabla (`clean.build_client_table`). Comprobamos que da exactamente lo mismo:

In [ ]:
pipeline_completo = pl.build_full_pipeline()
X_completo = pipeline_completo.fit_transform(clean.load_raw())
print("Forma:", X_completo.shape, "| Igual que por pasos:", np.allclose(X_completo, X))
pipeline_completo

Para el reto 3 también tenemos `pl.build_pipeline(k)`, que es este preprocesado con el K-Means detrás. Es justo lo que re-entrenaremos en el Jackknife.

## 9. Variante con telemetría (para el reto 3)

In [ ]:
prep_tele = pl.build_preprocessor(telemetry=True)
X_tele = prep_tele.fit_transform(clientes)
print("Forma de X con telemetría:", X_tele.shape)
print("Columnas extra:", [c for c in prep_tele.get_feature_names_out() if c not in nombres])

Cuatro columnas más. La distancia, el consumo y las horas llevan log y recorte; la velocidad solo recorte (no tiene cola larga, y con log empeora). Las compararemos en el reto 3.

## 10. Guardamos X

Guardamos la matriz del modelo base y los nombres de columnas para que el resto de notebooks no tenga que repetir nada:

In [ ]:
pl.save_X(X, nombres, clientes.index)
print("Guardado:", config.X_NPY.relative_to(config.ROOT), "y", config.FEATURE_NAMES_JSON.relative_to(config.ROOT))

## 11. Resumen: features del modelo

| Feature | Tratamiento | Por qué la usamos |
|---|---|---|
| `n_vehicles` | log + recorte + escalado | Tamaño de la relación con la marca; separa particulares de flotas |
| `n_connected_now` | log + recorte + escalado | Vehículos activos hoy; 0 = cliente sin relación activa |
| `population` | mediana del país + log + recorte + escalado | Zona urbana o rural |
| `workshop_visits_mean` | log + recorte + escalado | Relación con el taller → planes de mantenimiento |
| `ppi_rel` *(derivada)* | mediana del país + relativo al país + log + recorte + escalado | Riqueza de su zona dentro de su mercado |
| `age_ord` | recorte + escalado | Etapa de vida |
| `tenure_months` | mediana + recorte + escalado | Fidelidad / antigüedad con la marca |
| `main_vehicle_age` *(derivada)* | mediana + recorte + escalado | Cercanía a la renovación → ofertas de vehículo |
| `share_plugin_ev` *(derivada)* | mediana (0) + recorte + escalado | Afinidad con la electrificación |
| `is_ch` | 0/1 | Mercado (Suiza / Austria) |
| `is_b2b` | 0/1 | Empresa o particular |
| `tenure_is_default` *(derivada)* | 0/1 | Avisa de que su antigüedad es un valor por defecto |
| `has_telemetry` *(derivada)* | 0/1 | Cliente conectado o no |
| `main_class_group` *(derivada)* | one-hot (7) | Segmento de producto de su coche principal |
| `main_engine` *(derivada)* | one-hot (5) | Motorización de su coche principal |
| `finance_leasing_status` | one-hot (4) | Cómo compra: contado o leasing, activo o terminado |

**Fuera del modelo** (y por qué): género (inferido), consentimientos (dicen por dónde contactar, no quién es), `fleet_age_mean` (repite `main_vehicle_age`), `has_current_vehicle` (está dentro de `n_connected_now`), `ppi` en bruto (sustituido por `ppi_rel`), telemetría (falta en la mitad; se prueba como variante en el reto 3).

Las variables derivadas están explicadas con detalle en `02a_features_cliente`. Con esto ya tenemos `X` listo para el K-Means.